# Tarea 5: Heart Failure Prediction: Reducción de la Dimensionalidad - Parte 3. 

MDS7104: Aprendizaje de Maquinas - Otoño 2026

---

### Cuerpo Docente:

- Profesor: Francisco Vásquez L.
- Auxiliares: Álvaro Márquez y Diego Olguín Wende
- Ayudantes: Javiera Yañez y Tamara Carrasco


### Estudiante

- Felipe Muñoz M.

In [ ]:
!uv add numpy requests pandas matplotlib scipy scikit-learn ruff pre-commit

Resolved 63 packages in 20ms
Audited 58 packages in 16ms


In [3]:
!uv add ipykernel ipython matplotlib-inline

Resolved 130 packages in 24ms
Audited 127 packages in 26ms


In [2]:
import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

In [3]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
# %matplotlib inline

# Código Base Tarea 3 y 4

Consideraciones:
* Clase positiva `HeartDiseare=1`
* Se generan variables dummies con `drop_first=True` para evitar colinealidad perfecta
* Se fija la misma semilla (`seed=7`) y la misma partición 80/20 estratificada empleada en la Tarea 3, así los resultados de ambas tareas serán directamente comparables
* Se considera el siguiente conjunto de métricas: `M ={Accuracy, Precision, Recall, Specificity, F1-Score, FPR, FNR}`

In [4]:
# Semilla global
SEED = 7

# Carga de datos
df = pd.read_csv("/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/tareas/tarea3/heart.csv")

In [5]:
# Preprocesamiento: dummies con drop_first=True
df_dummies = pd.get_dummies(df, drop_first=True)

X = df_dummies.drop(columns="HeartDisease").values
y = df_dummies["HeartDisease"].values
feature_names = df_dummies.drop(columns="HeartDisease").columns.tolist()

# Partición 80/20 estratificada (misma semilla que T3/T4 -> resultados comparables)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

# ------------------------------------------------------------
# Identificación de columnas numéricas vs. dummy (binarias)

cols_dummy = [i for i, name in enumerate(feature_names) if df_dummies[name].nunique() == 2]
cols_num = [i for i in range(X.shape[1]) if i not in cols_dummy]
nombres_num = [feature_names[i] for i in cols_num]

# Estandarización: fit SOLO en train, transform en train y test (sin data leakage)
scaler = StandardScaler()
X_train_sc = X_train.copy().astype(float)
X_test_sc = X_test.copy().astype(float)
X_train_sc[:, cols_num] = scaler.fit_transform(X_train[:, cols_num])
X_test_sc[:, cols_num] = scaler.transform(X_test[:, cols_num])

# ------------------------------------------------------------
# Modelos baseline heredados

# Regresión Logística (mejor configuración hallada en Tarea 3)
lr_best = LogisticRegression(
    C=10,
    class_weight={0: 1, 1: 5},
    solver="saga",
    l1_ratio=1.0,
    max_iter=2000,
    random_state=SEED,
)
lr_best.fit(X_train, y_train)

# LDA (covarianza compartida, todas las variables)
lda = LinearDiscriminantAnalysis()
lda.fit(X_train, y_train)

# SVM con mejor configuración (Tarea 4): RBF, C=1, gamma='scale', class_weight=None
svc_best = SVC(
    kernel="rbf",
    C=1,
    gamma="scale",
    class_weight=None,
    random_state=SEED,
)
svc_best.fit(X_train_sc, y_train)

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'rbf'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [6]:
# Función de métricas M
def metricas_M(y_true, y_pred, nombre="") -> dict:
    """
    Calcula el conjunto M = {Accuracy, Precision, Recall, Specificity,
    F1-Score, FPR, FNR} e imprime un resumen.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metricas = {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "Specificity": tn / (tn + fp),
        "F1-Score": f1_score(y_true, y_pred, zero_division=0),
        "FPR": fp / (fp + tn),
        "FNR": fn / (fn + tp),
    }
    if nombre:
        print(f"\n── Métricas {nombre} ──────────────────────────")
        for k, v in metricas.items():
            print(f"  {k:<12}: {v:.4f}")
    return metricas

# P1 - Reducción de dimensionalidad lineal y supervisada

## a) Análisis de componentes principales

Convención (iv): el PCA se aplica sobre las variables numéricas estandarizadas del conjunto de entrenamiento.

In [7]:
X_train_num = X_train_sc[:, cols_num]
X_test_num = X_test_sc[:, cols_num]

M = X_train_num.shape[1]  # número de variables numéricas

# Ajuste de PCA con todas las componentes posibles
pca = PCA(n_components=M, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_num)
X_test_pca = pca.transform(X_test_num)

eigenvalues = pca.explained_variance_
var_explicada = pca.explained_variance_ratio_
var_acumulada = np.cumsum(var_explicada)

In [8]:
# 1) Reporte de los primeros valores propios
print("── Valores propios (varianza explicada) por componente ──")
for j, (lam, ve, vac) in enumerate(zip(eigenvalues, var_explicada, var_acumulada, strict=False), start=1):
    print(f"  PC{j:<2}: λ = {lam:7.4f}  |  VE = {ve:6.2%}  |  VE acumulada = {vac:6.2%}")

# Número de componentes para retener al menos 90% de varianza
d_90 = np.argmax(var_acumulada >= 0.90) + 1
print(f"\nComponentes necesarias para retener ≥90% de la varianza: d = {d_90}")
print(f"Varianza acumulada con d={d_90}: {var_acumulada[d_90-1]:.2%}")

── Valores propios (varianza explicada) por componente ──
  PC1 : λ =  1.7106  |  VE = 34.16%  |  VE acumulada = 34.16%
  PC2 : λ =  1.1880  |  VE = 23.73%  |  VE acumulada = 57.89%
  PC3 : λ =  0.8247  |  VE = 16.47%  |  VE acumulada = 74.36%
  PC4 : λ =  0.6997  |  VE = 13.97%  |  VE acumulada = 88.34%
  PC5 : λ =  0.5839  |  VE = 11.66%  |  VE acumulada = 100.00%

Componentes necesarias para retener ≥90% de la varianza: d = 5
Varianza acumulada con d=5: 100.00%


In [9]:
# 2) Gráfico: varianza explicada por componente + varianza acumulada
componentes = [f"PC{j}" for j in range(1, M + 1)]

fig_var = make_subplots(specs=[[{"secondary_y": True}]])

fig_var.add_trace(
    go.Bar(
        x=componentes, y=var_explicada,
        name="Varianza explicada",
        marker_color="steelblue",
    ),
    secondary_y=False,
)

fig_var.add_trace(
    go.Scatter(
        x=componentes, y=var_acumulada,
        name="Varianza acumulada",
        mode="lines+markers",
        line=dict(color="crimson", width=2),
    ),
    secondary_y=True,
)

fig_var.add_hline(
    y=0.90, line_dash="dash", line_color="gray",
    annotation_text="Umbral 90%", secondary_y=True,
)

fig_var.update_layout(
    title="PCA — Varianza explicada y varianza acumulada por componente",
    xaxis_title="Componente principal",
    width=750, height=450,
    legend=dict(x=0.65, y=0.5),
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig_var.update_yaxes(title_text="Varianza explicada", secondary_y=False, tickformat=".0%",
                      showgrid=True, gridcolor="lightgrey")
fig_var.update_yaxes(title_text="Varianza acumulada", secondary_y=True, tickformat=".0%",
                      range=[0, 1.05], showgrid=False)
fig_var.update_xaxes(showgrid=True, gridcolor="lightgrey")

fig_var.show()
fig_var.write_image("pca_varianza_explicada.png", scale=2)

In [10]:
# 3) Dispersión de las dos primeras componentes, coloreado por HeartDisease

df_pca2 = pd.DataFrame({
    "PC1": X_train_pca[:, 0],
    "PC2": X_train_pca[:, 1],
    "HeartDisease": y_train.astype(str),
})

fig_scatter = px.scatter(
    df_pca2, x="PC1", y="PC2", color="HeartDisease",
    color_discrete_map={"0": "steelblue", "1": "crimson"},
    title="PCA — Proyección sobre PC1 y PC2",
    labels={"PC1": f"PC1 ({var_explicada[0]:.1%})", "PC2": f"PC2 ({var_explicada[1]:.1%})"},
    opacity=0.7,
)
fig_scatter.update_layout(
    width=650, height=500,
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig_scatter.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig_scatter.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig_scatter.show()
fig_scatter.write_image("pca_scatter_pc1_pc2.png", scale=2)

In [11]:
# 4) Inspección de los vectores propios (loadings) de PC1 y PC2

loadings = pd.DataFrame(
    pca.components_[:2].T,
    columns=["PC1", "PC2"],
    index=nombres_num,
).sort_values("PC1", key=abs, ascending=False)

print("\n── Loadings (vectores propios) de PC1 y PC2 ──")
print(loadings.round(3).to_string())

fig_load = make_subplots(rows=1, cols=2, subplot_titles=("PC1", "PC2"))
fig_load.add_trace(
    go.Bar(x=loadings.index, y=loadings["PC1"], marker_color="steelblue", name="PC1"),
    row=1, col=1,
)
fig_load.add_trace(
    go.Bar(x=loadings.index, y=loadings["PC2"], marker_color="indianred", name="PC2"),
    row=1, col=2,
)
fig_load.update_layout(
    title="PCA — Contribución de variables originales a PC1 y PC2",
    width=850, height=400, showlegend=False,
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig_load.update_xaxes(tickangle=45, showgrid=True, gridcolor="lightgrey")
fig_load.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig_load.show()
fig_load.write_image("pca_loadings.png", scale=2)


── Loadings (vectores propios) de PC1 y PC2 ──
               PC1    PC2
Age          0.594  0.035
MaxHR       -0.539  0.347
Oldpeak      0.417  0.348
RestingBP    0.371  0.471
Cholesterol -0.210  0.732


## b) Kernel PCA

In [12]:
from sklearn.decomposition import KernelPCA

# Reutilizamos X_train_num / X_test_num (variables numéricas estandarizadas)

# 1) Proyección a 2D con kernels RBF y poly (parámetros por defecto)
kpca_rbf = KernelPCA(n_components=2, kernel="rbf", gamma=None, random_state=SEED)
X_train_kpca_rbf = kpca_rbf.fit_transform(X_train_num)

kpca_poly = KernelPCA(n_components=2, kernel="poly", degree=3, random_state=SEED)
X_train_kpca_poly = kpca_poly.fit_transform(X_train_num)

# Gráfico comparativo: PCA lineal vs Kernel PCA (rbf, poly)
fig_kpca = make_subplots(
    rows=1, cols=3,
    subplot_titles=("PCA lineal", "Kernel PCA (RBF, gamma=0.2 (default))", "Kernel PCA (poly, degree=3)"),
)

datasets = [
    (X_train_pca[:, :2], "PCA"),
    (X_train_kpca_rbf, "KPCA-RBF"),
    (X_train_kpca_poly, "KPCA-Poly"),
]

colores = {0: "steelblue", 1: "crimson"}

for col, (Xp, _) in enumerate(datasets, start=1):
    for clase in [0, 1]:
        mask = y_train == clase
        fig_kpca.add_trace(
            go.Scatter(
                x=Xp[mask, 0], y=Xp[mask, 1],
                mode="markers",
                marker=dict(color=colores[clase], size=5, opacity=0.6),
                name=f"HeartDisease={clase}",
                legendgroup=str(clase),
                showlegend=(col == 1),
            ),
            row=1, col=col,
        )

fig_kpca.update_layout(
    title="Comparación: PCA lineal vs. Kernel PCA (RBF y polinomial)",
    width=1100, height=400,
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig_kpca.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig_kpca.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig_kpca.show()
fig_kpca.write_image("kpca_comparacion_rbf_poly.png", scale=2)

In [13]:
# 2) Sensibilidad a gamma en kernel RBF (al menos 3 valores)
gammas = [0.01, 0.1, 1]

fig_gamma = make_subplots(
    rows=1, cols=len(gammas),
    subplot_titles=[f"gamma = {g}" for g in gammas],
)

for col, g in enumerate(gammas, start=1):
    kpca_g = KernelPCA(n_components=2, kernel="rbf", gamma=g, random_state=SEED)
    Xp = kpca_g.fit_transform(X_train_num)
    for clase in [0, 1]:
        mask = y_train == clase
        fig_gamma.add_trace(
            go.Scatter(
                x=Xp[mask, 0], y=Xp[mask, 1],
                mode="markers",
                marker=dict(color=colores[clase], size=5, opacity=0.6),
                name=f"HeartDisease={clase}",
                legendgroup=str(clase),
                showlegend=(col == 1),
            ),
            row=1, col=col,
        )

fig_gamma.update_layout(
    title="Kernel PCA (RBF) — Sensibilidad al parámetro gamma",
    width=1100, height=400,
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig_gamma.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig_gamma.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig_gamma.show()
fig_gamma.write_image("kpca_rbf_gamma.png", scale=2)

In [14]:
# 3) Sensibilidad al grado del polinomio en kernel poly (al menos 2 valores)
degrees = [2, 3, 5]

fig_degree = make_subplots(
    rows=1, cols=len(degrees),
    subplot_titles=[f"degree = {d}" for d in degrees],
)

for col, d in enumerate(degrees, start=1):
    kpca_d = KernelPCA(n_components=2, kernel="poly", degree=d, random_state=SEED)
    Xp = kpca_d.fit_transform(X_train_num)
    for clase in [0, 1]:
        mask = y_train == clase
        fig_degree.add_trace(
            go.Scatter(
                x=Xp[mask, 0], y=Xp[mask, 1],
                mode="markers",
                marker=dict(color=colores[clase], size=5, opacity=0.6),
                name=f"HeartDisease={clase}",
                legendgroup=str(clase),
                showlegend=(col == 1),
            ),
            row=1, col=col,
        )

fig_degree.update_layout(
    title="Kernel PCA (poly) — Sensibilidad al grado del polinomio",
    width=1100, height=400,
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig_degree.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig_degree.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig_degree.show()
fig_degree.write_image("kpca_poly_degree.png", scale=2)

## c) Fisher Discriminant

Ajustamos LDA sobre las variables numéricas estandarizadas del train (consistente con la convención (iv): mismas variables que PCA/KPCA)

In [15]:
lda_fisher = LinearDiscriminantAnalysis(n_components=1)
lda_fisher.fit(X_train_num, y_train)

# Proyección 1D (training)
X_train_fisher = lda_fisher.transform(X_train_num).ravel()

In [17]:
# 1) Dispersión unidimensional (strip plot) por clase, para comparar con PC1

fig_strip = make_subplots(
    rows=2, cols=1,
    subplot_titles=("Proyección de Fisher (LDA)", "Primera componente principal (PCA, PC1)"),
    shared_xaxes=False,
)

rng = np.random.RandomState(SEED)

for clase, color in [(0, "steelblue"), (1, "crimson")]:
    mask = y_train == clase

    # Fisher
    jitter_f = rng.uniform(-0.2, 0.2, size=mask.sum())
    fig_strip.add_trace(
        go.Scatter(
            x=X_train_fisher[mask], y=jitter_f,
            mode="markers", marker=dict(color=color, size=5, opacity=0.5),
            name=f"HeartDisease={clase}", legendgroup=str(clase), showlegend=True,
        ),
        row=1, col=1,
    )

    # PCA PC1
    jitter_p = rng.uniform(-0.2, 0.2, size=mask.sum())
    fig_strip.add_trace(
        go.Scatter(
            x=X_train_pca[mask, 0], y=jitter_p,
            mode="markers", marker=dict(color=color, size=5, opacity=0.5),
            name=f"HeartDisease={clase}", legendgroup=str(clase), showlegend=False,
        ),
        row=2, col=1,
    )

fig_strip.update_layout(
    title="Comparación de separación: Fisher (LDA) vs. PC1 (PCA)",
    width=750, height=550,
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig_strip.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig_strip.update_yaxes(showticklabels=False, showgrid=False, zeroline=False, range=[-1, 1])
fig_strip.show()
fig_strip.write_image("fisher_vs_pc1.png", scale=2)

In [18]:
# 2) Cuantificación de la separación: distancia de medias / dispersión

def separacion_J(proyeccion, y):
    """Calcula J = (m1-m2)^2 / (s1^2 + s2^2), criterio de Fisher 1D."""
    m0 = proyeccion[y == 0].mean()
    m1 = proyeccion[y == 1].mean()
    s0 = ((proyeccion[y == 0] - m0) ** 2).sum()
    s1 = ((proyeccion[y == 1] - m1) ** 2).sum()
    return (m1 - m0) ** 2 / (s0 + s1)

J_fisher = separacion_J(X_train_fisher, y_train)
J_pc1 = separacion_J(X_train_pca[:, 0], y_train)

print("── Criterio de Fisher J(a) = (m1-m0)^2 / (s0^2+s1^2) ──")
print(f"  Proyección de Fisher (LDA) : J = {J_fisher:.4f}")
print(f"  Primera componente (PC1)   : J = {J_pc1:.4f}")
print(f"\n  Razón J_fisher / J_pc1 = {J_fisher / J_pc1:.2f}x")

── Criterio de Fisher J(a) = (m1-m0)^2 / (s0^2+s1^2) ──
  Proyección de Fisher (LDA) : J = 0.0025
  Primera componente (PC1)   : J = 0.0019

  Razón J_fisher / J_pc1 = 1.36x


# P2 - Reducción de dimensionalidad no lineal y sensibilidad ahiperparámetros

## a) Aplicación inicial - Isomap, t-SNE y UMAP

In [20]:
import umap
from sklearn.manifold import TSNE, Isomap

Se Reutiliza `X_train_num` (variables numéricas estandarizadas del train)

In [21]:
# Ajuste de los tres métodos con parámetros "razonables" por defecto

isomap = Isomap(n_neighbors=15, n_components=2)
X_train_isomap = isomap.fit_transform(X_train_num)

tsne = TSNE(n_components=2, perplexity=30, random_state=SEED, init="pca")
X_train_tsne = tsne.fit_transform(X_train_num)

reducer_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, n_components=2, random_state=SEED)
X_train_umap = reducer_umap.fit_transform(X_train_num)

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [23]:
# Gráfico comparativo: Isomap, t-SNE, UMAP

fig_manifold = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Isomap", "t-SNE", "UMAP"),
)

datasets = [X_train_isomap, X_train_tsne, X_train_umap]
colores = {0: "steelblue", 1: "crimson"}

for col, Xp in enumerate(datasets, start=1):
    for clase in [0, 1]:
        mask = y_train == clase
        fig_manifold.add_trace(
            go.Scatter(
                x=Xp[mask, 0], y=Xp[mask, 1],
                mode="markers",
                marker=dict(color=colores[clase], size=5, opacity=0.6),
                name=f"HeartDisease={clase}",
                legendgroup=str(clase),
                showlegend=(col == 1),
            ),
            row=1, col=col,
        )

fig_manifold.update_layout(
    title="Proyecciones no lineales — Isomap, t-SNE y UMAP",
    width=1100, height=420,
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig_manifold.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig_manifold.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig_manifold.show()
fig_manifold.write_image("manifold_inicial.png", scale=2)

## b) Sensibilidad a hiperparámetros.

In [24]:
colores = {0: "steelblue", 1: "crimson"}

def plot_grid_proyecciones(proyecciones, titulos, titulo_general, filename, ncols=3):
    """Grafica una grilla de proyecciones 2D coloreadas por HeartDisease."""
    fig = make_subplots(rows=1, cols=ncols, subplot_titles=titulos)
    for col, Xp in enumerate(proyecciones, start=1):
        for clase in [0, 1]:
            mask = y_train == clase
            fig.add_trace(
                go.Scatter(
                    x=Xp[mask, 0], y=Xp[mask, 1],
                    mode="markers",
                    marker=dict(color=colores[clase], size=5, opacity=0.6),
                    name=f"HeartDisease={clase}",
                    legendgroup=str(clase),
                    showlegend=(col == 1),
                ),
                row=1, col=col,
            )
    fig.update_layout(
        title=titulo_general,
        width=1100, height=420,
        paper_bgcolor="white",
        plot_bgcolor="white",
    )
    fig.update_xaxes(showgrid=True, gridcolor="lightgrey")
    fig.update_yaxes(showgrid=True, gridcolor="lightgrey")
    fig.show()
    fig.write_image(filename, scale=2)
    return fig

In [26]:
# 1) Isomap — variar n_neighbors en {5, 15, 50}

isomap_neighbors = [5, 15, 50]
proyecciones_isomap = []

for k in isomap_neighbors:
    iso = Isomap(n_neighbors=k, n_components=2)
    proyecciones_isomap.append(iso.fit_transform(X_train_num))

plot_grid_proyecciones(
    proyecciones_isomap,
    titulos=[f"n_neighbors = {k}" for k in isomap_neighbors],
    titulo_general="Isomap — Sensibilidad a n_neighbors",
    filename="isomap_n_neighbors.png",
)

In [27]:
# 2) t-SNE — variar perplexity en {5, 30, 100}
tsne_perplexities = [5, 30, 100]
proyecciones_tsne = []

for p in tsne_perplexities:
    tsne_p = TSNE(n_components=2, perplexity=p, random_state=SEED, init="pca")
    proyecciones_tsne.append(tsne_p.fit_transform(X_train_num))

plot_grid_proyecciones(
    proyecciones_tsne,
    titulos=[f"perplexity = {p}" for p in tsne_perplexities],
    titulo_general="t-SNE — Sensibilidad a perplexity",
    filename="tsne_perplexity.png",
)

In [28]:
# 3) UMAP — Figura 1: variar n_neighbors en {5, 15, 50}, min_dist fijo

umap_neighbors = [5, 15, 50]
min_dist_fijo = 0.1
proyecciones_umap_nn = []

for k in umap_neighbors:
    reducer = umap.UMAP(n_neighbors=k, min_dist=min_dist_fijo, n_components=2, random_state=SEED)
    proyecciones_umap_nn.append(reducer.fit_transform(X_train_num))

plot_grid_proyecciones(
    proyecciones_umap_nn,
    titulos=[f"n_neighbors = {k}" for k in umap_neighbors],
    titulo_general=f"UMAP — Sensibilidad a n_neighbors (min_dist={min_dist_fijo} fijo)",
    filename="umap_n_neighbors.png",
)

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [29]:
# 4) UMAP — Figura 2: variar min_dist en {0.0, 0.3, 0.8}, n_neighbors fijo

umap_min_dists = [0.0, 0.3, 0.8]
n_neighbors_fijo = 15
proyecciones_umap_md = []

for md in umap_min_dists:
    reducer = umap.UMAP(n_neighbors=n_neighbors_fijo, min_dist=md, n_components=2, random_state=SEED)
    proyecciones_umap_md.append(reducer.fit_transform(X_train_num))

plot_grid_proyecciones(
    proyecciones_umap_md,
    titulos=[f"min_dist = {md}" for md in umap_min_dists],
    titulo_general=f"UMAP — Sensibilidad a min_dist (n_neighbors={n_neighbors_fijo} fijo)",
    filename="umap_min_dist.png",
)

/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
/Users/felijandro/Documents/Universidad/12voSemestre/AprendizajedeMaquinas/MDS7104-ML/.venv/lib/python3.12/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


# P3 - Reducción de dimensionalidad como paso previo a la clasificación

## a) Construcción de los modelos (Pipeline reducción + SVM)

In [30]:
from sklearn.decomposition import KernelPCA
from sklearn.pipeline import Pipeline

# Hiperparámetros ganadores del SVM en Tarea 4 (kernel rbf, C=1, gamma='scale')
SVM_C = 1
SVM_GAMMA = "scale"
SVM_CLASS_WEIGHT = None  # ganador en T4: class_weight=None

M = X_train_num.shape[1]  # número total de variables numéricas (M=5)

In [31]:
# Modelo de referencia: SVM RBF (mejor config T4) sobre variables originales

# Usamos svc_best ya entrenado en el setup, pero lo replicamos vía pipeline
modelo_referencia = Pipeline([
    ("svm", SVC(kernel="rbf", C=SVM_C, gamma=SVM_GAMMA,
                class_weight=SVM_CLASS_WEIGHT, random_state=SEED)),
])
modelo_referencia.fit(X_train_sc, y_train)

# ------------------------------------------------------------
# PCA(d) + SVM lineal, para d in {2, 5, M}

dims_pca = [2, 5, M]  # M=5 -> PCA con todas las componentes

modelos_pca = {}
for d in dims_pca:
    pipe = Pipeline([
        ("pca", PCA(n_components=d, random_state=SEED)),
        ("svm", SVC(kernel="linear", C=SVM_C,
                     class_weight=SVM_CLASS_WEIGHT, random_state=SEED)),
    ])
    pipe.fit(X_train_num, y_train)
    modelos_pca[f"PCA(d={d})+SVM_lin"] = pipe

# ------------------------------------------------------------
# Kernel PCA (rbf, d=5) + SVM lineal

modelo_kpca = Pipeline([
    ("kpca", KernelPCA(n_components=5, kernel="rbf", gamma=None, random_state=SEED)),
    ("svm", SVC(kernel="linear", C=SVM_C,
                 class_weight=SVM_CLASS_WEIGHT, random_state=SEED)),
])
modelo_kpca.fit(X_train_num, y_train)

# ------------------------------------------------------------
# Fisher Discriminant (LDA, d=1) + SVM lineal

modelo_fisher = Pipeline([
    ("lda", LinearDiscriminantAnalysis(n_components=1)),
    ("svm", SVC(kernel="linear", C=SVM_C,
                 class_weight=SVM_CLASS_WEIGHT, random_state=SEED)),
])
modelo_fisher.fit(X_train_num, y_train)

# ------------------------------------------------------------
# Diccionario unificado con los 6 modelos

modelos = {
    **modelos_pca,
    "KPCA(rbf,d=5)+SVM_lin": modelo_kpca,
    "Fisher(d=1)+SVM_lin": modelo_fisher,
    "SVM_RBF(referencia)": modelo_referencia,
}

print("── Modelos construidos ──")
for nombre in modelos:
    print(f"  - {nombre}")

── Modelos construidos ──
  - PCA(d=2)+SVM_lin
  - PCA(d=5)+SVM_lin
  - KPCA(rbf,d=5)+SVM_lin
  - Fisher(d=1)+SVM_lin
  - SVM_RBF(referencia)


## b) Resultados: Tabla de resultados (AUC, Accuracy, F1) + curvas ROC

In [32]:
# Diccionario de datos de test por modelo (según las variables que usa cada pipeline)
X_test_map_p3 = {
    "PCA(d=2)+SVM_lin": X_test_num,
    "PCA(d=5)+SVM_lin": X_test_num,
    f"PCA(d={M})+SVM_lin": X_test_num,
    "KPCA(rbf,d=5)+SVM_lin": X_test_num,
    "Fisher(d=1)+SVM_lin": X_test_num,
    "SVM_RBF(referencia)": X_test_sc,
}

X_train_map_p3 = {
    "PCA(d=2)+SVM_lin": X_train_num,
    "PCA(d=5)+SVM_lin": X_train_num,
    f"PCA(d={M})+SVM_lin": X_train_num,
    "KPCA(rbf,d=5)+SVM_lin": X_train_num,
    "Fisher(d=1)+SVM_lin": X_train_num,
    "SVM_RBF(referencia)": X_train_sc,
}

# Cálculo de métricas y scores para cada modelo
resultados_p3 = {}
roc_data = {}

for nombre, pipe in modelos.items():
    X_te = X_test_map_p3[nombre]

    y_pred = pipe.predict(X_te)
    scores = pipe.decision_function(X_te)  # SVM con kernel lineal/rbf -> decision_function

    auc_val = roc_auc_score(y_test, scores)
    acc_val = accuracy_score(y_test, y_pred)
    f1_val = f1_score(y_test, y_pred, zero_division=0)

    resultados_p3[nombre] = {
        "AUC": auc_val,
        "Accuracy": acc_val,
        "F1-Score": f1_val,
    }

    fpr, tpr, _ = roc_curve(y_test, scores)
    roc_data[nombre] = (fpr, tpr, auc_val)

In [37]:
# Tabla de resultados

df_resultados_p3 = pd.DataFrame(resultados_p3).T.round(4)
df_resultados_p3 = df_resultados_p3[["AUC", "Accuracy", "F1-Score"]]

print("── Tabla de resultados comparación de modelos ──")
print(df_resultados_p3.to_string())

── Tabla de resultados comparación de modelos ──
                          AUC  Accuracy  F1-Score
PCA(d=2)+SVM_lin       0.7622    0.7065    0.7404
PCA(d=5)+SVM_lin       0.8274    0.7772    0.7960
KPCA(rbf,d=5)+SVM_lin  0.8190    0.7283    0.7685
Fisher(d=1)+SVM_lin    0.8269    0.7663    0.7882
SVM_RBF(referencia)    0.9408    0.8804    0.8972


In [ ]:
# Curvas ROC superpuestas (6 modelos)

fig_roc = go.Figure()

palette = px.colors.qualitative.Set1

for i, (nombre, (fpr, tpr, auc_val)) in enumerate(roc_data.items()): # noqa: B007
    fig_roc.add_trace(
        go.Scatter(
            x=fpr, y=tpr,
            mode="lines",
            name=f"{nombre}",
            line=dict(color=palette[i % len(palette)], width=2),
        )
    )

# Línea de referencia (clasificador aleatorio)
fig_roc.add_trace(
    go.Scatter(
        x=[0, 1], y=[0, 1],
        mode="lines",
        name="Clasificador aleatorio",
        line=dict(color="gray", width=1.5, dash="dash"),
    )
)

fig_roc.update_layout(
    title="Curvas ROC — Comparación de modelos",
    xaxis_title="Tasa de Falsos Positivos (FPR)",
    yaxis_title="Tasa de Verdaderos Positivos (TPR)",
    width=750, height=550,
    legend=dict(x=0.55, y=0.05),
    paper_bgcolor="white",
    plot_bgcolor="white",
)
fig_roc.update_xaxes(showgrid=True, gridcolor="lightgrey")
fig_roc.update_yaxes(showgrid=True, gridcolor="lightgrey")
fig_roc.show()
fig_roc.write_image("p3_roc_comparacion.png", scale=2)